# Esercizi sulle distribuzioni discrete

* Durante la pandemia molti esami furono sostituiti da test online a risposta multipla. Se un test prevede 5 domande, ciascuna con 3 risposte, calcolare la probabilità di indovinare almeno 4 risposte a caso. Come cambia la probabilità se invece ci sono 5 risposte?

**Soluzione:**

* $N$ = numero di domande
* $R$ = numero di risposte per ciascuna domanda
* $p=1/R$ = probabilità di indovinare a caso una domanda
* Il numero $n$ di risposte esatte segue la distribuzione binomiale con $P(n; p=1/R, N)$
* La probabilità di indovinare almeno $K$ risposte è $\pi = \sum_{n=K, \cdots, N}P(n; p=1/R, N) = 1 - C(K-1; p=1/R, N)$

In [7]:
import numpy as np
from scipy.stats import binom
N = 5
R = 3
K = 4
def prob_rnd():
    p = 1/R
    b = binom(5, p)
    return 1 - b.cdf(K - 1)
prob = prob_rnd()
print(f"almeno {K} domande, {R} risposte: p = {prob:.2%}")
R = 4
prob = prob_rnd()
print(f"almeno {K} domande, {R} risposte: p = {prob:.2%}")

almeno 4 domande, 3 risposte: p = 4.53%
almeno 4 domande, 4 risposte: p = 1.56%


* In una produzione industriale viene riscontrato lo 0.1% di componenti difettosi. Calcolare la probabilità di averne almeno 2 difettosi in uno scatolo con 1000 componenti. Ripetere il calcolo utilizzando la distribuzione di Poisson.

**Soluzione:**
La distribuzione è binominale con $N=1000$ e $p=0.001$; si può approssimare con una poissoniana con $\nu=Np=1$.

In [2]:
from scipy.stats import binom, poisson
N = 1000
p = 0.001
nu = N * p
b = binom(N, p)
p = poisson(nu)
prob_binom = 1 - b.cdf(1)
prob_poisson = 1 - p.cdf(1)
print(f"binomiale:   p = {prob_binom:.8%}")
print(f"poissoniana: p = {prob_poisson:.8%}")

binomiale:   p = 26.42410870%
poissoniana: p = 26.42411177%


* Supponiamo che in un paese la probabilità che una spedizione tracciabile vada persa sia il 3 per mille. Calcolare la probabilità che un venditore su interenet abbia almeno 3 feedback negativi su 500 acquisti per colpa delle poste. Ripetere il calcolo assumendo distribuzione di probabilità binominale e poissoniana.

**Soluzione:**
La distribuzione è binominale con $N=500$ e $p=0.003$; si può approssimare con una poissoniana con $\nu=Np=1.5$.

In [3]:
from scipy.stats import binom, poisson
N = 500
p = 0.003
nu = N * p
b = binom(N, p)
p = poisson(nu)
prob_binom = 1 - b.cdf(2)
prob_poisson = 1 - p.cdf(2)
print(f"binomiale:   p = {prob_binom:.8%}")
print(f"poissoniana: p = {prob_poisson:.8%}")

binomiale:   p = 19.09642348%
poissoniana: p = 19.11531695%


* Una ditta produce monitor 4K (3840 x 2160 pixel). Calcolare quale deve essere la probabilità massima di avere un pixel morto se la ditta deve garantire di avere il 99% dei monitor con al massimo un pixel morto. Si può approssimare la distribuzione binomiale con quella di Poisson?

**Soluzione:**

In [25]:
from scipy.optimize import bisect

px, py = 3840, 2160
n_pixels = px * py
print("n. pixel, 4K: ", n_pixels)
p_min = 0.99
px_dead_max = 1

def prob_monitor(p_1px_dead):
    p = poisson(n_pixels*p_1px_dead)
    prob_poiss = p.cdf(1)
    # b = binom(n_pixels, p_1px_dead)
    # prob_binom = b.cdf(1)
    # print(f"media n. px morti: {b.mean()} (binom.), {p.mean()} (poiss.)")
    return prob_poiss

for p_1px_dead in np.linspace(1e-8, 2e-8, 5):
    p_mon = prob_monitor(p_1px_dead)
    print(f"p(singolo px morto) = {p_1px_dead:.4e}:\n"
          f"p(4K, <= 1px morti) = {p_mon:.4%} (binom.)")

p_1px = bisect(lambda x: prob_monitor(x) - p_min, 0, 1)
print(f"p ({p_min:.2%}) = {p_1px:4e}")

n. pixel, 4K:  8294400
p(singolo px morto) = 1.0000e-08:
p(4K, <= 1px morti) = 99.6745% (binom.)
p(singolo px morto) = 1.2500e-08:
p(4K, <= 1px morti) = 99.4983% (binom.)
p(singolo px morto) = 1.5000e-08:
p(4K, <= 1px morti) = 99.2873% (binom.)
p(singolo px morto) = 1.7500e-08:
p(4K, <= 1px morti) = 99.0431% (binom.)
p(singolo px morto) = 2.0000e-08:
p(4K, <= 1px morti) = 98.7672% (binom.)
p (99.00%) = 1.791159e-08


* In un anno si stima che circa 17000 meteoriti colpiscano la terra con una massa da qualche grammo a mezzo chilo[*] . Calcolare la probabilità che entro vent'anni cada almeno un meteorite sul terrazzo di copertura di un edificio. Assumere una superficie di 300 metri quadri.

$[*]$ [Meteorites on Earth: how many fall per year and why don't we see them?](https://www.iberdrola.com/innovation/meteorites-earth)

**Soluzione:**

Per probabilità piccole, la probabilità di avere 0 conteggi in un processo poissoniano è:

$e^{-\nu} \simeq 1 - \nu + ...$

Quindi la probabilità di avere 1 o più conteggi è:

$1 - e^{-\nu} \simeq \nu$

In [34]:
n_meteor_year_earth = 17000
n_years = 20
surf_building = 300 # m²
surf_eargh = 5.1e14 # m²
r_surf = surf_building/surf_eargh
n_meteor = n_meteor_year_earth * r_surf * 20
print(f"mean n. meteor = {n_meteor}")
p_meteor = 1 - poisson.pmf(0, n_meteor)
print(f"p >= 1. meteor = {p_meteor:.3e}")

mean n. meteor = 2e-07
p >= 1. meteor = 2.000e-07


* In un prato cadono 25mm di pioggia, che vengono misurati con un pluviometro che ha una base di 10cm$^2$. Calcolare la varianza e la deviazione standard della misura, assumendo che le gocce di pioggia abbiano tutte un volume di 0.1ml[*] e una precisione molto alta della misura del volume totale di acqua.

$[*]$ [Rain drop size and speed of a falling rain drop](https://www.baranidesign.com/faq-articles/2020/1/19/rain-drop-size-and-speed-of-a-falling-rain-drop)

**Soluzione:**

Il numero di gocce fluttuerà rispetto al valore atteso $\nu$, e la deviazione standard è $\sigma=\sqrt{\nu}$.

La misura del volume, trascurando le altre incertezze, è proporzionale a $\nu$, quindi è affetta da un errore
relativo $r = \sigma/\nu=\sqrt{\nu}/\nu = 1/\sqrt{\nu}$.

In [49]:
import math

base = 1e-3 # m²
height_exp = 2.5e-2 # m
vol_drop = 1e-7 # m^3
vol_exp = base * height_exp
n_drop_exp = vol_exp / vol_drop
n_drop_std = math.sqrt(n_drop_exp)
print(f"n. atteso di gocce: {n_drop_exp:.2f} +/- {n_drop_std:.2f}")
r = n_drop_std / n_drop_exp
print(f"err. relativo: {r:.3%}")
height_std = height_exp * r
print(f"h attesa: {height_exp:.5f} +/- {height_std:.5f} m")

n. atteso di gocce: 250.00 +/- 15.81
err. relativo: 6.325%
h attesa: 0.02500 +/- 0.00158 m


* In Italia nel 2019 per incidenti d'auto sono morte 3.173 persone e 241.384 sono risultate ferite (fonte: [ISTAT](https://www.istat.it/it/archivio/245757)) su un totale di [59.816.673 abitanti](http://demo.istat.it/popres/index.php?anno=2019). Qual è la probabilità che a Pozzuoli (79.752 abitanti) ci siano meno di 2 morti in un anno, assumendo una probabilità uniforme sul territorio nazionale?

**Soluzione:**

"Stimo" la probabilità di morire in Italia in un anno come $3.173/59.816.673$. Questo non è il valore vero, ma una stima!
Da notare quindi che la stima del numero atteso di morti a Pozzuoli è affetto almeno dall'incertezza relativa pari a $1/\sqrt{3173}$. Ci saranno poi altre incertezze legate alle assunzioni fatte, es.: la probabilità di morte è unforme, ecc.

In [64]:
n_dead_ita = 3173
n_ita = 59816673
n_pozz = 79753
p_ita = n_dead_ita/n_ita
print(f"prob. di morte: {p_ita:.4%}")
n_pozz_exp = p_ita * n_pozz
print(f"n. attesi: {n_pozz_exp:.2f}")
p_less_2_pozz = poisson.cdf(1, n_pozz_exp)
print(f"prob < 2 morti: {p_less_2_pozz:.3%}")

prob. di morte: 0.0053%
n. attesi: 4.23
prob < 2 morti: 7.608%


* Il 17/3/2021 il Paul-Erlich-Institut ha circolato un [comunicato](https://www.pei.de/SharedDocs/Downloads/EN/newsroom-en/hp-news/faq-temporary-suspension-astrazeneca.pdf) che ha portato alla sospenzione del vaccino AstraZeneca. Erano stati osservati 7 casi (su 1.7 milioni) di una rara trombosi quandi ne era atteso, in media 1. Calcolare qual è la probabilità di avere 7 o più casi quando ne sono attesi 1. Calcolare la stessa probabilità quando se ne attendono invece 2 o 3.

* Si vuole misurare l'efficienza di un rivelatore contando il numero di particelle $n$ che vengono rivelate rispetto ad un numero $N$ di particelle che lo attraversano. Assumendo che l'efficienza sia vicina al 90%, calcolare i segnali di quante particelle $N$ vanno acquisiti in modo da avere una deviazione standard del valore misurato intorno all'1%.

**Soluzione:**

La deviazione standard di una binomiale con $p=\varepsilon$ è $\sigma_n = \sqrt{N\varepsilon(1-\varepsilon)}$.

Quindi, se $\varepsilon = n/N$ (vero solo nel limite $N\rightarrow\infty$) ho $\sigma_\varepsilon = \sqrt{\varepsilon(1-\varepsilon)/N}$

In [72]:
def err_epsilon(N, epsilon):
    return math.sqrt(epsilon*(1 - epsilon)/N)

epsilon = 0.90
err_target = 0.01
N_best = bisect(lambda x: err_epsilon(x, epsilon) - err_target, 100, 1000)
print(f"N best: {N_best:.2f}")

N best: 900.00


* Si vuole misurare l'intensità dei fasci di un collider di elettroni e positroni[*] contando in numero di coppie elettrone-positrone prodotti attravreso un rivelatore posto a piccolo angolo. In un anno vengono acquisti $N=123456$ eventi di questo tipo. Stimare l'incertezza relativa nella misura di intensità come rapporto tra deviazione standard e valore atteso.

$[*]$ tecnicamente, questa si chiama misura di *luminosità integrata*

**Soluzione:**

L'errore relativo di un processo poissoniano è sempre $1/\sqrt{\nu}$. In questo caso: $1/\sqrt{123456}$

In [76]:
print(f"errore relatovo: {1/math.sqrt(123456):.2%}")

errore relatovo: 0.28%


* Ad una certa data, in Italia 1.200.000 persone erano state identificate come infette (e in isolamento) da Covid-19. Assumiamo che altrettante persone siano infette e circolassero tranquillamente senza sapere di essere infetti. In un treno ci sono 200 persone. Trovare la probabilità che almeno 4 siano infette.

* Un rivelatore posto in un esperimento su un acceleratore è attraversato da $\nu=100$ particelle in media per ogni collissione registrata. Considerando che il sistema di acquisizione riesce ad acquisire segnali di $N^\star=150$ particelle contemporaneamente, al massimo, calcolare la frazione di collisioni per la quale il sistema di aquisizione va in "saturazione". Si assuma, per semplicità, che il numero di particelle che attraversa il rivelatore segua una distribuzione di Poisson, anche se questo non è necessariamente vero.

$[*]$ questa frazione si chiama tecnicamente frazione di *tempo morto* del sistema di acquisizione.

**Soluzione:**

La deviazione standard di una poissoniana con $\nu=100$ è $\sigma = \sqrt{\nu} =10$. 
Avere 50 conteggi in più devia di 5 deviazioni standard rispetto al valore atteso.

In [79]:
print(f"prob. saturazione: {1 - poisson.cdf(150, 100):.3e}")

prob. saturazione: 1.233e-06


* Un rivelatore di particelle è fatto da 5 piani di rivelatori piani, ciascuno con efficienza del 90%. Consideriamo di identificare una particella se, attraversando tutti i 5 piani, dà un segnale su almeno 3 dei 5 rivelatori. Calcolare l'efficienza di questa selezione.

Questo modo di identificare le particelle prende il nome di *majority*.

**Soluzione:**

Il numero di rivelatori che danno un segnale segue la distribuzione binomiale con $N=5$ e $p=0.9$.

In [81]:
N = 5
p = 0.9
n_min = 3
prob = 1 - binom(N, p).cdf(n_min - 1)
print(f"prob. majority {n_min}/{N} = {prob:.2%}")

prob. majority 3/5 = 99.14%
